In [3]:
!pip install \
  torch_geometric \
  pyg_lib torch_scatter \
  torch_sparse \
  torch_cluster \
  torch_spline_conv \
  -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install imbalanced-learn \
  tqdm

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html


In [4]:
"""
Simplified QAT GNN Model for Cyber Threat Detection
---------------------------------------------------
Uses a minimal approach with standard PyG operations and careful quantization
to ensure compatibility with quantized backends.
"""

import os
import pickle
import random
import copy
from typing import Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.quantization import (
    get_default_qat_qconfig,
    prepare_qat,
    convert,
    QuantStub,
    DeQuantStub
)
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn.utils import clip_grad_norm_
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv, BatchNorm
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from google.colab import drive

# Set backend for server CPUs (x86)
torch.backends.quantized.engine = 'fbgemm'

# Paths
drive.mount('/content/drive', force_remount=True)
DRIVE_PATH = '/content/drive/MyDrive/CyberThreatDetectionSystem_Project/'
DATA_PATH = os.path.join(DRIVE_PATH, 'Data', 'processed')
MODEL_PATH = os.path.join(DRIVE_PATH, 'Models')
os.makedirs(MODEL_PATH, exist_ok=True)


def set_seed(seed: int = 50):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_graph_data() -> Tuple:
    """Load train/val/test graph data from pickles."""
    def _load(split: str):
        path = os.path.join(DATA_PATH, f"{split}_graph.pkl")
        with open(path, 'rb') as f:
            return pickle.load(f)
    return _load('train'), _load('val'), _load('test')


def process_graph_data(train_raw, val_raw, test_raw) -> Tuple:
    """Convert raw data to PyG Data objects with normalized features."""
    # Use robust normalization based on training data
    feats = torch.tensor(train_raw['x'], dtype=torch.float32)
    q_low, q_high = torch.quantile(feats, torch.tensor([0.01, 0.99]), dim=0)
    iqr = torch.where(q_high - q_low > 1e-6, q_high - q_low, torch.ones_like(q_low))

    def to_data(raw_graph):
        x = torch.tensor(raw_graph['x'], dtype=torch.float32)
        x = torch.clamp((x - q_low) / iqr, -5.0, 5.0)
        return Data(
            x=x,
            edge_index=torch.tensor(raw_graph['edge_index'], dtype=torch.long),
            y=torch.tensor(raw_graph['y'], dtype=torch.long)
        )

    return to_data(train_raw), to_data(val_raw), to_data(test_raw)


def create_loaders(train_data, val_data, test_data,
                  batch_size: int = 4096,
                  neighbors: List[int] = [50, 40, 30, 20]):
    """Create data loaders for train/val/test sets."""
    return (
        NeighborLoader(train_data, num_neighbors=neighbors, batch_size=batch_size, shuffle=True),
        NeighborLoader(val_data, num_neighbors=neighbors, batch_size=batch_size, shuffle=False),
        NeighborLoader(test_data, num_neighbors=neighbors, batch_size=batch_size, shuffle=False)
    )


class SimpleQATGNN(nn.Module):
    """
    A simplified GNN model that's compatible with quantization.
    Uses standard PyTorch operations and avoids complex custom operations.
    """
    def __init__(self, in_channels, hidden_dim=192, num_layers=4):
        super().__init__()
        self.quant = QuantStub()
        self.dequant = DeQuantStub()

        # Initial projection
        self.input_proj = nn.Linear(in_channels, hidden_dim)
        self.relu = nn.ReLU(inplace=True)

        # GNN layers - using standard PyG SAGEConv
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(BatchNorm(hidden_dim))

        # Output layers
        self.fc1 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc2 = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x, edge_index):
        # Quantize input features
        x = self.quant(x)

        # Initial projection
        x = self.input_proj(x)
        x = self.relu(x)

        # GNN layers
        for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
            identity = x
            # Use the standard SAGEConv which is more likely to be quantization-compatible
            x = conv(x, edge_index)
            x = bn(x)
            x = self.relu(x)
            x = x + identity  # Residual connection

        # Final classification head
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        # Dequantize output
        x = self.dequant(x)

        return x.squeeze()


def train_qat_model(epochs=150, batch_size=4096):
    """
    Train with QAT and evaluate performance.
    """
    print("Setting up data and models...")
    set_seed(50)
    device = 'cpu'

    # Load and process data
    train_raw, val_raw, test_raw = load_graph_data()
    train_data, val_data, test_data = process_graph_data(train_raw, val_raw, test_raw)
    train_loader, val_loader, test_loader = create_loaders(
        train_data, val_data, test_data, batch_size=batch_size
    )

    # Class balance weights
    neg_count = (train_data.y == 0).sum().item()
    pos_count = (train_data.y == 1).sum().item()
    pos_weight = torch.tensor([1.33 * neg_count / pos_count])
    print(f"Class balance – Normal: {neg_count}, Attack: {pos_count}")

    # Create QAT model
    print("Creating QAT model...")
    qat_model = SimpleQATGNN(train_data.x.size(1), hidden_dim=192, num_layers=4)
    qat_model.train()  # Set to training mode

    # Prepare the model for QAT
    print("Preparing model for quantization-aware training...")
    qat_model.qconfig = get_default_qat_qconfig('fbgemm')
    prepare_qat(qat_model, inplace=True)

    # Set up optimizer and scheduler
    optimizer = AdamW(qat_model.parameters(), lr=5e-4, weight_decay=3.5e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.75, patience=8)

    # Training loop
    best_f1, patience, best_thr = 0.0, 0, 0.5
    best_model_weights = None
    print("\n===== QAT Training =====")

    for epoch in tqdm(range(1, epochs + 1), desc="QAT Epoch"):
        qat_model.train()
        epoch_loss = 0.0

        # Learning rate warmup
        if epoch < 5:
            for pg in optimizer.param_groups:
                pg['lr'] = 5e-4 * (epoch / 5)

        # Train for one epoch
        for batch in train_loader:
            x_batch = batch.x
            e_idx = batch.edge_index
            logits = qat_model(x_batch, e_idx)[:batch.batch_size]
            target = batch.y[:batch.batch_size].float()

            # Label smoothing
            target = target * 0.99 + 0.005

            loss = F.binary_cross_entropy_with_logits(
                logits, target, pos_weight=pos_weight
            )

            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(qat_model.parameters(), max_norm=2.5)
            optimizer.step()
            epoch_loss += loss.item()

        # Validation
        qat_model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                scores = torch.sigmoid(qat_model(batch.x, batch.edge_index)[:batch.batch_size])
                val_preds.append(scores.cpu())
                val_labels.append(batch.y[:batch.batch_size].cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_labels = torch.cat(val_labels).numpy()

        # Find best threshold
        thresholds = np.linspace(0.3, 0.7, 41)
        f1_scores = [f1_score(val_labels, val_preds > thr) for thr in thresholds]
        best_idx = int(np.argmax(f1_scores))
        current_f1 = f1_scores[best_idx]
        current_thr = float(thresholds[best_idx])

        print(f"Epoch {epoch}: loss={epoch_loss:.4f} — Val F1={current_f1:.4f}@thr={current_thr:.3f}")
        scheduler.step(current_f1)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thr = current_thr
            best_model_weights = copy.deepcopy(qat_model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= 25:
                print(f"Early stopping at epoch {epoch}")
                break


    # Load best weights
    print("Loading best model weights...")
    qat_model.load_state_dict(best_model_weights)
    qat_model.eval()

    print("\n===== Final Evaluation =====")

    # Evaluate on test set before quantization
    test_preds, test_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            scores = torch.sigmoid(qat_model(batch.x, batch.edge_index)[:batch.batch_size])
            test_preds.append(scores.cpu())
            test_labels.append(batch.y[:batch.batch_size].cpu())

    test_preds = torch.cat(test_preds).numpy()
    test_labels = torch.cat(test_labels).numpy()
    qat_f1 = f1_score(test_labels, test_preds > best_thr)
    qat_precision = precision_score(test_labels, test_preds > best_thr)
    qat_recall = recall_score(test_labels, test_preds > best_thr)

    print(f"QAT Model Test Metrics:")
    print(f"F1 Score   : {qat_f1:.6f}")
    print(f"Precision  : {qat_precision:.6f}")
    print(f"Recall     : {qat_recall:.6f}")
    print(f"Threshold  : {best_thr:.6f}")

    # Convert to fully quantized model
    print("Converting to fully quantized model...")
    try:
        # Try to convert the model
        quantized_model = convert(copy.deepcopy(qat_model).cpu(), inplace=False)

        # Save the quantized model
        torch.save({
            'model_state_dict': quantized_model.state_dict(),
            'threshold': best_thr,
            'f1': qat_f1,
            'precision': qat_precision,
            'recall': qat_recall
        }, os.path.join(MODEL_PATH, 'quantized_model.pth'))

        print("Quantized model saved successfully!")
        return quantized_model, best_thr

    except Exception as e:
        # If conversion fails, save the QAT model
        print(f"Notice: Quantization conversion requires special handling.")

        # Save the QAT model
        torch.save({
            'model_state_dict': qat_model.state_dict(),
            'threshold': best_thr,
            'f1': qat_f1,
            'precision': qat_precision,
            'recall': qat_recall
        }, os.path.join(MODEL_PATH, 'qat_model.pth'))

        return qat_model, best_thr


if __name__ == '__main__':
    model, threshold = train_qat_model()

Mounted at /content/drive
Setting up data and models...
Class balance – Normal: 7012, Attack: 46435
Creating QAT model...
Preparing model for quantization-aware training...


/usr/local/lib/python3.11/dist-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(



===== QAT Training =====


QAT Epoch:   0%|          | 0/150 [00:00<?, ?it/s]

Epoch 1: loss=2.5418 — Val F1=0.9117@thr=0.490
Epoch 2: loss=1.6041 — Val F1=0.9632@thr=0.600
Epoch 3: loss=0.9751 — Val F1=0.9828@thr=0.700
Epoch 4: loss=0.7261 — Val F1=0.9865@thr=0.540
Epoch 5: loss=0.6434 — Val F1=0.9873@thr=0.300
Epoch 6: loss=0.5847 — Val F1=0.9873@thr=0.610
Epoch 7: loss=0.5659 — Val F1=0.9893@thr=0.310
Epoch 8: loss=0.5505 — Val F1=0.9869@thr=0.300
Epoch 9: loss=0.5036 — Val F1=0.9896@thr=0.300
Epoch 10: loss=0.4762 — Val F1=0.9896@thr=0.310
Epoch 11: loss=0.4617 — Val F1=0.9928@thr=0.300
Epoch 12: loss=0.4717 — Val F1=0.9913@thr=0.300
Epoch 13: loss=0.4500 — Val F1=0.9914@thr=0.300
Epoch 14: loss=0.4256 — Val F1=0.9922@thr=0.300
Epoch 15: loss=0.4245 — Val F1=0.9839@thr=0.300
Epoch 16: loss=0.4744 — Val F1=0.9846@thr=0.300
Epoch 17: loss=0.5265 — Val F1=0.9867@thr=0.300
Epoch 18: loss=0.4427 — Val F1=0.9933@thr=0.510
Epoch 19: loss=0.4195 — Val F1=0.9925@thr=0.370
Epoch 20: loss=0.4175 — Val F1=0.9911@thr=0.690
Epoch 21: loss=0.4155 — Val F1=0.9936@thr=0.690
E